04 - Feature Engineering: Bag of Words

## Setup
Run the cell below first. It detects whether you're in **Google Colab** or
running **locally in VS Code**, and gets the environment ready either way
(clones the repo in Colab, installs requirements, downloads NLTK data, and
adds `src/` to the path so `pipeline.py` can be imported).

In [1]:
# ============================================================
# SETUP CELL - run this first, every time
# Works both locally (VS Code / Jupyter) and in Google Colab
# ============================================================
import os, sys
import shutil # Added for removing directories

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/Sandaru17513/NLP_Ctrl-Alt-Elite.git"
    REPO_DIR = "NLP_Ctrl-Alt-Elite"

    # Always ensure we are in the root /content directory before any operations
    os.chdir('/content')

    # Remove existing directory to ensure a clean clone and prevent nesting issues
    if os.path.exists(REPO_DIR):
        print(f"Removing existing '{REPO_DIR}' directory to ensure a clean clone.")
        shutil.rmtree(REPO_DIR)

    get_ipython().system(f"git clone {REPO_URL}")

    # Now change to the correct notebooks directory within the cloned repo
    target_notebooks_dir = os.path.join('/content', REPO_DIR, "notebooks")
    if os.path.exists(target_notebooks_dir):
        os.chdir(target_notebooks_dir)
    else:
        print(f"Error: Notebooks directory not found at {target_notebooks_dir}")
        # You might want to raise an exception or handle this more robustly

    # Install requirements, including langdetect
    get_ipython().system("pip install -q -r ../requirements.txt langdetect")

    # Data files are large - if they were not committed to the repo,
    # upload them here once per Colab session.
    if not os.path.exists("../data/data.csv"):
        print("data/data.csv not found in the cloned repo.")
        print("Option A: git add + commit + push the CSVs from your")
        print("          local machine so they come down with the clone.")
        print("Option B: uncomment the lines below to upload manually.")
        # from google.colab import files
        # uploaded = files.upload()   # select data.csv + validation_dataset.csv
        # os.makedirs("../data", exist_ok=True)
        # for fname in uploaded:
        #     os.rename(fname, f"../data/{fname}")
else:
    print("Running locally (VS Code / Jupyter). Using existing .venv environment.")

import nltk
for pkg in ("punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

# sys.path should now correctly resolve ../src because the CWD is fixed
sys.path.append(os.path.abspath("../src"))
print("IN_COLAB =", IN_COLAB)
print("Working directory:", os.getcwd())


Cloning into 'NLP_Ctrl-Alt-Elite'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 85 (delta 26), reused 74 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 33.32 MiB | 9.21 MiB/s, done.
Resolving deltas: 100% (26/26), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 10.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
data/data.csv not found in the cloned repo.
Option A: git add + commit + push the CSVs from your
          local machine so they come down with the clone.
Option B: uncomment the lines below to upload manually.
IN_COLAB = True
Working directory: /content/NLP_Ctrl-Alt-Elite/notebooks


### Final tokenization: stop-word removal + lemmatization

In [2]:
import sys, os

PIPELINE_SUBDIR = 'cit-24-01-0182'

# Adjust sys.path to include the specific subdirectory where pipeline.py resides
pipeline_module_path = os.path.abspath(os.path.join('../src', PIPELINE_SUBDIR))
sys.path.append(pipeline_module_path)

# Verify if pipeline.py exists before attempting import
pipeline_file_path = os.path.join(pipeline_module_path, 'pipeline.py')

if not os.path.exists(pipeline_file_path):
    print(f"Error: The 'pipeline.py' file was not found at '{pipeline_file_path}'.")
    print("Please ensure it exists in the 'src/cit-24-01-0182' directory of your cloned repository.")
else:
    import pandas as pd
    from pipeline import tokenize_and_lemmatize

    # Check for 'preprocessed.csv' before attempting to read it
    preprocessed_csv_path = '../data/preprocessed.csv'
    if not os.path.exists(preprocessed_csv_path):
        print(f"Error: '{preprocessed_csv_path}' not found. Please ensure this data file exists.")
    else:
        df = pd.read_csv(preprocessed_csv_path)
        df['final_text'] = df['normalized_text'].apply(tokenize_and_lemmatize)

        print('Before:', df['normalized_text'].iloc[0][:150])
        print('After: ', df['final_text'].iloc[0][:150])

        # Ensure output directory exists before writing CSV
        output_dir = os.path.dirname('../data/final_processed.csv')
        os.makedirs(output_dir, exist_ok=True)
        df.to_csv('../data/final_processed.csv', index=False)


Error: '../data/preprocessed.csv' not found. Please ensure this data file exists.


### Upload Missing Data Files

The previous error indicates that `preprocessed.csv` (and likely other data files) are not present in the `../data` directory. You will need to upload these files manually.

**Expected Files:**
- `data.csv` (original dataset)
- `preprocessed.csv` (derived from `data.csv`)
- `validation_dataset.csv` (original validation dataset)
- `validation_preprocessed.csv` (derived from `validation_dataset.csv`)

Please execute the cell below and select these files from your local machine to upload them. Make sure to upload all four files if you have them.

After successfully uploading the files, please re-run the feature engineering cell (`86d7965c`).

In [3]:
import os

data_dir = '../data'

if os.path.exists(data_dir) and os.path.isdir(data_dir):
    print(f"Contents of '{data_dir}':")
    if not os.listdir(data_dir):
        print("  (Directory is empty)")
    else:
        for item in os.listdir(data_dir):
            print(f"- {item}")
else:
    print(f"Error: The directory '{data_dir}' does not exist or is not a directory.")

Contents of '../data':
- dataset.csv
- .gitkeep
- validation_dataset.csv


### Generate `preprocessed.csv` and `validation_preprocessed.csv`

Since `preprocessed.csv` and `validation_preprocessed.csv` are missing, and you have `dataset.csv` and `validation_dataset.csv`, we will generate the intermediate preprocessed files. This cell assumes that your `pipeline.py` module contains a function (e.g., `preprocess_text`) that can take raw text and produce a normalized version, which is then stored in a `normalized_text` column.

In [9]:
import pandas as pd
import os
import sys

# Ensure the path to pipeline.py is in sys.path
PIPELINE_SUBDIR = 'cit-24-01-0182'
pipeline_module_path = os.path.abspath(os.path.join('../src', PIPELINE_SUBDIR))
if pipeline_module_path not in sys.path:
    sys.path.append(pipeline_module_path)

# Dynamically import the pipeline module
try:
    # Attempt to import the pipeline module
    import pipeline
    # Check if normalize_text function exists within the imported module
    if hasattr(pipeline, 'normalize_text'): # FIX: Changed from preprocess_text to normalize_text
        preprocess_text = pipeline.normalize_text # FIX: Assign normalize_text to preprocess_text for consistency with downstream code
    else:
        raise ImportError(f"'normalize_text' function not found in {pipeline_module_path}/pipeline.py")
except ImportError as e:
    print(f"Error importing 'normalize_text' from pipeline.py: {e}") # FIX: Changed from preprocess_text to normalize_text
    print(f"Please ensure 'pipeline.py' exists in '{pipeline_module_path}' and defines a 'normalize_text' function.") # FIX: Changed from preprocess_text to normalize_text
    # Fallback/dummy function to prevent further errors if not found
    def preprocess_text(text):
        print("Warning: Using dummy preprocess_text function. Please check your pipeline.py.")
        return str(text) if pd.notna(text) else ''

# Load raw datasets
try:
    raw_df = pd.read_csv('../data/dataset.csv')
    raw_val_df = pd.read_csv('../data/validation_dataset.csv')
except FileNotFoundError as e:
    print(f"Error loading raw data: {e}. Please ensure dataset.csv and validation_dataset.csv are in ../data/")
    raise

# Apply preprocessing assuming 'text' is the raw text column for the main dataset
df_preprocessed = raw_df.copy()
if 'text' in df_preprocessed.columns:
    df_preprocessed['normalized_text'] = df_preprocessed['text'].apply(preprocess_text)
else:
    print("Warning: 'text' column not found in dataset.csv. 'normalized_text' column will not be created.")

# Apply preprocessing for validation dataset, using 'Email Text' column
val_df_preprocessed = raw_val_df.copy()
if 'Email Text' in val_df_preprocessed.columns: # FIX: Changed 'text' to 'Email Text'
    val_df_preprocessed['normalized_text'] = val_df_preprocessed['Email Text'].apply(preprocess_text) # FIX: Changed 'text' to 'Email Text'
else:
    print("Warning: 'Email Text' column not found in validation_dataset.csv. 'normalized_text' column will not be created.")


# Save preprocessed files
os.makedirs('../data', exist_ok=True)
df_preprocessed.to_csv('../data/preprocessed.csv', index=False)
val_df_preprocessed.to_csv('../data/validation_preprocessed.csv', index=False)

print("Created '../data/preprocessed.csv' and '../data/validation_preprocessed.csv'")
print("Displaying head of preprocessed_df:")
display(df_preprocessed.head())

Created '../data/preprocessed.csv' and '../data/validation_preprocessed.csv'
Displaying head of preprocessed_df:


,label,text,normalized_text
0,1,Congratulations! You've been selected for a lu...,congratulations you ve been selected for a lux...
1,1,URGENT: Your account has been compromised. Cli...,urgent your account has been compromised click...
2,1,You've won a free iPhone! Claim your prize by ...,you ve won a free iphone claim your prize by c...
3,1,Act now and receive a 50% discount on all purc...,act now and receive a discount on all purchase...
4,1,Important notice: Your subscription will expir...,important notice your subscription will expire...


In [10]:
import sys, os

PIPELINE_SUBDIR = 'cit-24-01-0182'

# Adjust sys.path to include the specific subdirectory where pipeline.py resides
pipeline_module_path = os.path.abspath(os.path.join('../src', PIPELINE_SUBDIR))
sys.path.append(pipeline_module_path)

# Verify if pipeline.py exists before attempting import
pipeline_file_path = os.path.join(pipeline_module_path, 'pipeline.py')

if not os.path.exists(pipeline_file_path):
    print(f"Error: The 'pipeline.py' file was not found at '{pipeline_file_path}'.")
    print("Please ensure it exists in the 'src/cit-24-01-0182' directory of your cloned repository.")
else:
    import pandas as pd
    from pipeline import tokenize_and_lemmatize

    # Check for 'preprocessed.csv' before attempting to read it
    preprocessed_csv_path = '../data/preprocessed.csv'
    if not os.path.exists(preprocessed_csv_path):
        print(f"Error: '{preprocessed_csv_path}' not found. Please ensure this data file exists.")
    else:
        df = pd.read_csv(preprocessed_csv_path)
        df['final_text'] = df['normalized_text'].apply(tokenize_and_lemmatize)

        print('Before:', df['normalized_text'].iloc[0][:150])
        print('After: ', df['final_text'].iloc[0][:150])

        # Ensure output directory exists before writing CSV
        output_dir = os.path.dirname('../data/final_processed.csv')
        os.makedirs(output_dir, exist_ok=True)
        df.to_csv('../data/final_processed.csv', index=False)

Before: congratulations you ve been selected for a luxury vacation getaway claim your prize now
After:  congratulation selected luxury vacation getaway claim prize


After running the above cell successfully, please:
1. Re-run the feature engineering cell (`86d7965c`).
2. Re-run the train/test split cell (`40c57691`).

In [11]:
import os

src_dir = '../src'

if os.path.exists(src_dir) and os.path.isdir(src_dir):
    print(f"Contents of '{src_dir}':")
    for item in os.listdir(src_dir):
        print(f"- {item}")
else:
    print(f"Error: The directory '{src_dir}' does not exist or is not a directory.")


Contents of '../src':
- .gitkeep
- cit-24-01-0182


In [12]:
PIPELINE_SUBDIR = 'cit-24-01-0182'
pipeline_module_path = os.path.abspath(os.path.join('../src', PIPELINE_SUBDIR))
pipeline_file_path = os.path.join(pipeline_module_path, 'pipeline.py')

if os.path.exists(pipeline_file_path):
    print(f"Contents of {pipeline_file_path}:")
    !cat "{pipeline_file_path}"
else:
    print(f"Error: The file '{pipeline_file_path}' does not exist.")

Contents of /content/NLP_Ctrl-Alt-Elite/src/cit-24-01-0182/pipeline.py:
"""
src/pipeline.py
Member 1 -- P. Chamika Janith Piyasena (CIT-24-01-0182)
Group 6 -- Spam Email/SMS Detection System

Reusable NLP preprocessing pipeline for Member 1's unique steps:
    Language Detection -> Sentence Segmentation -> Text Normalization
    -> Tokenization / Stop-word removal / Lemmatization

Import this module from any notebook OR from the group's final web
application so preprocessing is 100% consistent between training and
inference (avoids train/serve skew).

Usage:
    from pipeline import full_pipeline, detect_language, segment_sentences

    clean_text = full_pipeline(raw_email_text)
"""

import re
import nltk
from langdetect import detect, LangDetectException
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ---------------------------------------------------------------------------
# One-time NLTK downloads 

### Train/test split + CountVectorizer (fit on train only!)

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
import joblib
import os

df = pd.read_csv('../data/final_processed.csv')

X = df['final_text'].fillna('')
y = df['label'].astype(int)   # spam = 1, ham = 0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)}')
print(f'Test size: {len(X_test)}')
print(f'Train spam: {y_train.sum()} ({y_train.mean()*100:.1f}%)')

bow_vectorizer = CountVectorizer(
    max_features=10000,
    ngram_range=(1, 2),   # unigrams + bigrams e.g. "click here", "free prize"
    min_df=2,
    max_df=0.95,
)

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

print(f'BoW feature matrix shape: {X_train_bow.shape}')

os.makedirs('../models', exist_ok=True)
joblib.dump(bow_vectorizer, '../models/bow_vectorizer.pkl')
joblib.dump((X_train, X_test, y_train, y_test), '../models/train_test_split.pkl')
print('Saved: bow_vectorizer.pkl, train_test_split.pkl')

Train size: 36124
Test size: 9031
Train spam: 18057 (50.0%)
BoW feature matrix shape: (36124, 10000)
Saved: bow_vectorizer.pkl, train_test_split.pkl


### Prepare the validation set with the SAME fitted vectorizer

In [14]:
df_val = pd.read_csv('../data/validation_preprocessed.csv')
df_val['final_text'] = df_val['Email Text'].apply(tokenize_and_lemmatize)

X_val = df_val['final_text'].fillna('')
y_val = (df_val['Email Type'] == 'Phishing Email').astype(int) # Map 'Phishing Email' to 1 (spam), 'Safe Email' to 0 (ham)

X_val_bow = bow_vectorizer.transform(X_val)   # transform only - never re-fit

print(f'Validation BoW shape: {X_val_bow.shape}')
print(f'Validation spam: {y_val.sum()} ({y_val.mean()*100:.1f}%)')

joblib.dump((X_val, X_val_bow, y_val), '../models/validation_bow.pkl')
print('Saved: validation_bow.pkl')

Validation BoW shape: (2000, 10000)
Validation spam: 1000 (50.0%)
Saved: validation_bow.pkl
